In [1]:
import numpy as np
import pandas as pd
import json
from collections import Counter
import cProfile
from pstats import SortKey
import pstats
from time import perf_counter

In [2]:
df = pd.read_json('../data/candidates.jsonl', lines=True)  

In [6]:
sample_df = df.sample(n=120, random_state=42)
sample_df.to_json('../data/candidates_sample_120.jsonl', orient='records', lines=True)
sample_df.head()

,candidate_id,profile,career_history,education,skills,certifications,languages,redrob_signals
75721,CAND_0075722,"{'anonymized_name': 'Aisha Goyal', 'headline':...","[{'company': 'Acme Corp', 'title': 'Cloud Engi...","[{'institution': 'IISc Bangalore', 'degree': '...","[{'name': 'MLflow', 'proficiency': 'advanced',...","[{'name': 'Scrum Master Certified', 'issuer': ...","[{'language': 'English', 'proficiency': 'profe...","{'profile_completeness_score': 55.1, 'signup_d..."
80184,CAND_0080185,"{'anonymized_name': 'Riya Shetty', 'headline':...","[{'company': 'Wayne Enterprises', 'title': 'Ac...","[{'institution': 'Amity University', 'degree':...","[{'name': 'Redis', 'proficiency': 'beginner', ...","[{'name': 'Scrum Master Certified', 'issuer': ...","[{'language': 'English', 'proficiency': 'profe...","{'profile_completeness_score': 42.2, 'signup_d..."
19864,CAND_0019865,"{'anonymized_name': 'Saanvi Singh', 'headline'...","[{'company': 'Stark Industries', 'title': 'Pro...","[{'institution': 'VJTI Mumbai', 'degree': 'B.S...","[{'name': 'Angular', 'proficiency': 'beginner'...",[],"[{'language': 'English', 'proficiency': 'profe...","{'profile_completeness_score': 74.5, 'signup_d..."
76699,CAND_0076700,"{'anonymized_name': 'Saanvi Arora', 'headline'...","[{'company': 'Stark Industries', 'title': 'Bus...","[{'institution': 'Bharati Vidyapeeth', 'degree...","[{'name': 'Go', 'proficiency': 'intermediate',...",[],"[{'language': 'English', 'proficiency': 'profe...","{'profile_completeness_score': 50.9, 'signup_d..."
92991,CAND_0092992,"{'anonymized_name': 'Neha Saxena', 'headline':...","[{'company': 'Acme Corp', 'title': '.NET Devel...","[{'institution': 'Amity University', 'degree':...","[{'name': 'Project Management', 'proficiency':...",[],"[{'language': 'English', 'proficiency': 'profe...","{'profile_completeness_score': 85.8, 'signup_d..."


In [7]:
# Create a dataset excluding the sample data
exclude_ids = set(sample_df['candidate_id'])
filtered_df = df[~df['candidate_id'].isin(exclude_ids)].reset_index(drop=True)

# Save the filtered dataset
filtered_df.to_json('../data/candidates_excluding_sample.jsonl', orient='records', lines=True)

print(f"Original dataset size: {len(df)}")
print(f"Sample dataset size: {len(sample_df)}")
print(f"Filtered dataset size: {len(filtered_df)}")
print(f"Filtered dataset saved to '../data/candidates_excluding_sample.jsonl'")

Original dataset size: 100000
Sample dataset size: 120
Filtered dataset size: 99880
Filtered dataset saved to '../data/candidates_excluding_sample.jsonl'


In [3]:
jsonl_path = '../data/candidates_excluding_sample.jsonl'

In [4]:
WRONG_TITLE_KEYWORDS = [
    "marketing", "sales", "hr ", "human resource", "recruiter", "talent acquisition",
    "finance", "accountant", "accounting", "business development", "bd manager",
    "supply chain", "logistics", "operations manager", "project manager",
    "product manager",
    "graphic design", "ux designer", "ui designer", "visual design",
    "content writer", "copywriter", "seo", "social media",
    "civil engineer", "mechanical engineer", "electrical engineer",
    "hardware engineer", "embedded", "firmware",
    "computer vision", "cv engineer",
    "speech", "audio", "robotics", "autonomous",
    "teacher", "professor", "lecturer", "faculty",
    "doctor", "physician", "nurse", "pharmacist",
    "lawyer", "legal", "compliance",
    "customer success", "customer support", "support engineer",
]

# ── Consulting firms from the JD's explicit disqualifier list ─────────────────
PURE_CONSULTING_FIRMS = {
    "tcs", "tata consultancy", "infosys", "wipro", "accenture",
    "cognizant", "capgemini", "hcl", "tech mahindra", "mphasis",
    "hexaware", "mindtree", "ltimindtree", "l&t infotech",
    "persistent systems", "niit technologies", "mastech",
    "kforce", "syntel", "unison",
}

INDIA_VARIANTS = {"india", "in", "ind"}

MIN_YOE = 3
MAX_YOE = 12

In [5]:
# ── Helpers ───────────────────────────────────────────────────────────────────

def matches_any(text, keywords):
    return any(kw in text.lower() for kw in keywords)

def is_consulting_firm(company_name):
    c = company_name.lower().strip()
    return any(firm in c for firm in PURE_CONSULTING_FIRMS)

def is_pure_consulting_career(career_history):
    if not career_history:
        return False
    return all(is_consulting_firm(role.get("company", "")) for role in career_history)

def is_job_hopper(career_history):
    if not career_history or len(career_history) < 2:
        return False, ""
    durations = [r.get("duration_months") or 0 for r in career_history]
    avg_tenure = sum(durations) / len(durations)
    companies = [r.get("company", "").strip().lower() for r in career_history]
    switches = len(set(companies))
    if avg_tenure < 18 and switches >= 3:
        return True, f"avg_tenure={avg_tenure:.1f}mo, switches={switches}"
    return False, ""

def is_title_chaser(career_history):
    TITLE_LEVELS = {
        "junior": 1, "associate": 1, "intern": 1, "trainee": 1,
        "engineer": 2, "developer": 2, "analyst": 2, "scientist": 2,
        "senior": 3, "sr.": 3, "sr ": 3, "lead": 3,
        "staff": 4, "specialist": 4, "expert": 4,
        "principal": 5, "architect": 5,
        "director": 6, "vp": 6, "head": 6, "manager": 6,
    }

    def title_level(title):
        t = title.lower()
        matched = [lvl for kw, lvl in TITLE_LEVELS.items() if kw in t]
        return max(matched) if matched else 0

    sorted_career = sorted(career_history, key=lambda r: r.get("start_date") or "0000-00-00")
    jumps = 0
    prev_level = 0
    for role in sorted_career:
        level = title_level(role.get("title", ""))
        duration = role.get("duration_months") or 0
        if level > prev_level and prev_level != 0 and duration < 18:
            jumps += 1
        prev_level = max(prev_level, level)

    if jumps >= 2:
        titles = [r.get("title", "") for r in sorted_career]
        return True, f"quick_jumps={jumps}, titles={titles}"
    return False, ""

In [6]:
# ── Pipeline Stage 1: Exclusion Filters ───────────────────────────────────────

def apply_exclusion_filters(candidates):
    """Applies hard constraints, plausibility checks, and structural limits."""
    passed_candidates = []
    
    stats = {
        "country_counter": Counter(),
        "yoe_distribution": Counter(),
        "consulting_firm_counter": Counter(),
        "non_india_ids": [],
        "too_junior_ids": [],
        "too_senior_ids": [],
        "null_skills_ids": [],
        "honeypot_skill_ids": [],
        "pure_consulting_ids": [],
        "job_hopper_ids": [],
        "title_chaser_ids": [],
        "total_processed": 0
    }

    for c in candidates:
        stats["total_processed"] += 1
        p = c["profile"]
        cid = c["candidate_id"]
        career = c.get("career_history") or []
        country = p.get("country", "").strip()
        yoe = p.get("years_of_experience", 0) or 0
        skills = c.get("skills") or []

        bucket = (int(yoe) // 2) * 2
        stats["yoe_distribution"][bucket] += 1

        exclude_reasons = []

        # Filter: country
        stats["country_counter"][country] += 1
        if country.lower() not in INDIA_VARIANTS:
            willing = c.get("redrob_signals", {}).get("willing_to_relocate", False)
            if not willing:
                stats["non_india_ids"].append((cid, country))
                exclude_reasons.append("non_india")

        # Filter: years of experience extremes
        if yoe < MIN_YOE:
            stats["too_junior_ids"].append((cid, yoe))
            exclude_reasons.append("too_junior")
        elif yoe > MAX_YOE:
            stats["too_senior_ids"].append((cid, yoe))
            exclude_reasons.append("too_senior")

        # Filter: null / empty skills
        if len(skills) <= 5:
            stats["null_skills_ids"].append(cid)
            exclude_reasons.append("null_skills")

        # Filter: honeypot skill signal
        if skills:
            expert_skills = [s for s in skills if s.get("proficiency") == "expert"]
            if expert_skills and all((s.get("duration_months") or 0) == 0 for s in expert_skills):
                stats["honeypot_skill_ids"].append(
                    (cid, [(s["name"], s.get("duration_months")) for s in expert_skills])
                )
                exclude_reasons.append("honeypot_skill")

        # Filter: pure consulting career
        for role in career:
            company = role.get("company", "")
            if is_consulting_firm(company):
                stats["consulting_firm_counter"][company] += 1

        if is_pure_consulting_career(career):
            companies = [r.get("company", "") for r in career]
            stats["pure_consulting_ids"].append((cid, companies))
            exclude_reasons.append("pure_consulting")

        # Filter: job hopper
        hopper, hopper_reason = is_job_hopper(career)
        if hopper:
            stats["job_hopper_ids"].append((cid, hopper_reason))
            exclude_reasons.append("job_hopper")

        # Filter: title chaser
        chaser, chaser_reason = is_title_chaser(career)
        if chaser:
            stats["title_chaser_ids"].append((cid, chaser_reason))
            exclude_reasons.append("title_chaser")

        # If no exclusions fired, pass candidate to next stage
        if not exclude_reasons:
            passed_candidates.append(c)

    return passed_candidates, stats

In [7]:
# ── Pipeline Stage 2: Keyword Pruning ─────────────────────────────────────────

def apply_keyword_pruning(candidates):
    """Filters out candidates based on disqualifying keywords in their titles."""
    passed_candidates = []
    
    stats = {
        "title_counter": Counter(),
        "wrong_title_ids": [],
        "total_processed": 0
    }

    for c in candidates:
        stats["total_processed"] += 1
        title = c["profile"].get("current_title", "").strip()
        cid = c["candidate_id"]
        
        stats["title_counter"][title] += 1
        
        # Filter: Wrong Title Keyword
        if matches_any(title, WRONG_TITLE_KEYWORDS):
            stats["wrong_title_ids"].append((cid, title))
        else:
            passed_candidates.append(c)

    return passed_candidates, stats

In [8]:
from datetime import date

MAX_INACTIVE_DAYS = 180
MIN_RESPONSE_RATE = 0.10

def get_dynamic_as_of_date(all_candidates):
    """Finds the most recent last_active_date across the entire candidate pool."""
    max_date_str = None
    for c in all_candidates:
        la = c.get("redrob_signals", {}).get("last_active_date")
        if la:
            if max_date_str is None or la > max_date_str:
                max_date_str = la
    
    if max_date_str:
        return date.fromisoformat(max_date_str[:10])
    return date.today() # Fallback if no dates exist


def apply_behavioral_pruning(candidates, as_of_date):
    """Drops candidates who are behaviorally unreachable (Inactive > 180d or RR < 10%)."""
    passed_candidates = []
    stats = {
        "inactive_ids": [],
        "low_response_ids": [],
        "total_processed": 0
    }

    for c in candidates:
        stats["total_processed"] += 1
        cid = c["candidate_id"]
        sig = c.get("redrob_signals", {})
        
        # 1. Check Inactivity
        la_str = sig.get("last_active_date")
        if not la_str:
            # None treated as worst-case (inactive > 6 months)
            stats["inactive_ids"].append((cid, "None"))
            continue
            
        la_date = date.fromisoformat(la_str[:10])
        days_inactive = (as_of_date - la_date).days
        
        if days_inactive > MAX_INACTIVE_DAYS:
            stats["inactive_ids"].append((cid, days_inactive))
            continue

        # 2. Check Response Rate
        rr = sig.get("recruiter_response_rate")
        if rr is None or rr < MIN_RESPONSE_RATE:
            stats["low_response_ids"].append((cid, rr))
            continue

        passed_candidates.append(c)

    return passed_candidates, stats

In [9]:
import re

# ── Configuration for Stage 4 ─────────────────────────────────────────────────

DOMAIN_KEYWORD_GROUPS = {
    "core_ml": {
        "machine learning", "deep learning", "nlp", "natural language", 
        "llm", "large language", "rag", "retrieval augmented"
    },
    "search_ranking": {
        "search", "retrieval", "ranking", "recommendation", "recommender",
        "information retrieval", "semantic search", "hybrid search"
    },
    "embeddings_models": {
        "embedding", "vector", "sentence-transformer", "bge", "e5", 
        "bert", "transformer"
    },
    "vector_dbs": {
        "pinecone", "weaviate", "qdrant", "milvus", "opensearch", 
        "elasticsearch", "faiss", "annoy", "scann"
    },
    "evaluation": {
        "ndcg", "mrr", "a/b test", "offline eval", "online eval",
        "precision@", "recall@"
    },
    "production_systems": {
        "production", "deployed", "inference", "latency", "throughput"
    }
}

# Pre-compile regex patterns for lightning-fast matching across descriptions
COMPILED_DOMAIN_GROUPS = {}
for group, keywords in DOMAIN_KEYWORD_GROUPS.items():
    patterns = []
    for kw in keywords:
        # Add word boundaries to avoid false positives (e.g., "rag" matching "storage")
        if kw[-1].isalnum():
            patterns.append(r"\b" + re.escape(kw) + r"\b")
        else:
            patterns.append(r"\b" + re.escape(kw))
            
    # Compile as a single regex OR statement for the entire group
    COMPILED_DOMAIN_GROUPS[group] = re.compile("|".join(patterns))

In [10]:
# ── Pipeline Stage 4: Domain Relevance (Depth Gate) ───────────────────────────

def apply_domain_relevance(candidates, min_required_groups=2):
    """
    Ensures the candidate is in the AI/ML/Search domain.
    Checks BOTH Skills and Career History Descriptions.
    Requires hits in at least `min_required_groups` distinct domain categories.
    """
    passed_candidates = []
    stats = {
        "irrelevant_domain_ids": [],
        "total_processed": 0
    }

    for c in candidates:
        stats["total_processed"] += 1
        cid = c["candidate_id"]
        
        # 1. Extract skills
        skills = c.get("skills", [])
        skill_names = " ".join([s.get("name", "").lower() for s in skills])
        
        # 2. Extract career history descriptions
        career = c.get("career_history", [])
        descriptions = " ".join([role.get("description", "").lower() for role in career])
        
        # 3. Combine into a single searchable text block
        searchable_text = f" {skill_names} {descriptions} "
        
        # 4. Count how many DISTINCT knowledge groups the candidate hits
        matched_groups_count = 0
        for group_name, pattern in COMPILED_DOMAIN_GROUPS.items():
            if pattern.search(searchable_text):
                matched_groups_count += 1
        
        # 5. Filter based on depth (Spread across groups = genuine capability)
        if matched_groups_count < min_required_groups:
            stats["irrelevant_domain_ids"].append(cid)
        else:
            passed_candidates.append(c)

    return passed_candidates, stats

In [11]:
# 1. Load data
all_candidates = []
with open(jsonl_path) as f:
    for line in f:
        line = line.strip()
        if line:
            all_candidates.append(json.loads(line))

In [12]:
# ============================================================
# PROFILER START
# ============================================================

profiler = cProfile.Profile()
profiler.enable()

stage_times = {}

# ============================================================
# PIPELINE
# ============================================================

# --- DYNAMIC DATE CALCULATION ---
t0 = perf_counter()
dynamic_as_of_date = get_dynamic_as_of_date(all_candidates)
stage_times["Dynamic Date"] = perf_counter() - t0

print(f"\n[*] Computed AS_OF_DATE for this pool: {dynamic_as_of_date}")

# ------------------------------------------------------------
# Stage 1
# ------------------------------------------------------------
t0 = perf_counter()

candidates_s1, exclusion_stats = apply_exclusion_filters(all_candidates)

stage_times["Stage 1 - Exclusion"] = perf_counter() - t0

# ------------------------------------------------------------
# Stage 2
# ------------------------------------------------------------
t0 = perf_counter()

candidates_s2, pruning_stats = apply_keyword_pruning(candidates_s1)

stage_times["Stage 2 - Keyword"] = perf_counter() - t0

# ------------------------------------------------------------
# Stage 3
# ------------------------------------------------------------
t0 = perf_counter()

candidates_s3, behavioral_stats = apply_behavioral_pruning(
    candidates_s2,
    dynamic_as_of_date,
)

stage_times["Stage 3 - Behavioral"] = perf_counter() - t0

# ------------------------------------------------------------
# Stage 4
# ------------------------------------------------------------
t0 = perf_counter()

final_candidates, domain_stats = apply_domain_relevance(candidates_s3)

stage_times["Stage 4 - Domain"] = perf_counter() - t0

# ============================================================
# PROFILER STOP
# ============================================================

profiler.disable()

# ============================================================
# STAGE TIMINGS
# ============================================================

print("\n" + "=" * 80)
print("STAGE EXECUTION TIMES")
print("=" * 80)

total_pipeline_time = sum(stage_times.values())

for stage, elapsed in stage_times.items():
    print(f"{stage:<30}: {elapsed:8.3f} sec")

print("-" * 80)
print(f"{'TOTAL PIPELINE':<30}: {total_pipeline_time:8.3f} sec")

# ============================================================
# cProfile REPORT
# ============================================================

print("\n" + "=" * 80)
print("TOP 30 FUNCTIONS BY CUMULATIVE TIME")
print("=" * 80)

stats = pstats.Stats(profiler)
stats.sort_stats(SortKey.CUMULATIVE)
stats.print_stats(30)

# Save detailed profile (can inspect later with snakeviz)
stats.dump_stats("candidate_pipeline.prof")

# ── Print Output Summary ──────────────────────────────────────────────────

total = exclusion_stats["total_processed"]

print("=" * 60)
print(f"TOTAL CANDIDATES (Initial): {total:,}")
print("=" * 60)

# Calculate overall unique excluded count from Stages 1 & 2
s1_s2_excluded_ids = (
    set(cid for cid, _ in exclusion_stats["non_india_ids"]) |
    set(cid for cid, _ in exclusion_stats["too_junior_ids"]) |
    set(cid for cid, _ in exclusion_stats["too_senior_ids"]) |
    set(exclusion_stats["null_skills_ids"]) |
    set(cid for cid, _ in exclusion_stats["honeypot_skill_ids"]) |
    set(cid for cid, _ in exclusion_stats["pure_consulting_ids"]) |
    set(cid for cid, _ in exclusion_stats["job_hopper_ids"]) |
    set(cid for cid, _ in exclusion_stats["title_chaser_ids"]) |
    set(cid for cid, _ in pruning_stats["wrong_title_ids"])
)

print(f"\n── STAGE 1 & 2: STRUCTURAL & KEYWORD EXCLUSIONS ──")
print(f"  [Stage 1] Non-India (no relocation): {len(exclusion_stats['non_india_ids']):>6,}")
print(f"  [Stage 1] Too junior (< {MIN_YOE} yoe):       {len(exclusion_stats['too_junior_ids']):>6,}")
print(f"  [Stage 1] Too senior (> {MAX_YOE} yoe):      {len(exclusion_stats['too_senior_ids']):>6,}")
print(f"  [Stage 1] Null/empty skills:         {len(exclusion_stats['null_skills_ids']):>6,}")
print(f"  [Stage 1] Honeypot skill signal:     {len(exclusion_stats['honeypot_skill_ids']):>6,}")
print(f"  [Stage 1] Pure consulting career:    {len(exclusion_stats['pure_consulting_ids']):>6,}")
print(f"  [Stage 1] Job hopper:                {len(exclusion_stats['job_hopper_ids']):>6,}")
print(f"  [Stage 1] Title chaser:              {len(exclusion_stats['title_chaser_ids']):>6,}")
print(f"  [Stage 2] Wrong title:               {len(pruning_stats['wrong_title_ids']):>6,}")
print(f"  ─────────────────────────────────────────")
print(f"  Unique Excluded (Stages 1 & 2):      {len(s1_s2_excluded_ids):>6,}  ({len(s1_s2_excluded_ids)/total*100:.1f}%)")
print(f"  Candidates Surviving to Stage 3:     {len(candidates_s2):>6,}")

print(f"\n── STAGE 3 & 4: BEHAVIORAL & DOMAIN CUTS ──")
print(f"  [Stage 3] Inactive > {MAX_INACTIVE_DAYS} days:      {len(behavioral_stats['inactive_ids']):>6,}")
print(f"  [Stage 3] Response Rate < {MIN_RESPONSE_RATE*100:.0f}%:       {len(behavioral_stats['low_response_ids']):>6,}")
print(f"  [Stage 4] Lacking Core ML/Search Skills: {len(domain_stats['irrelevant_domain_ids']):>6,}")
print(f"  ─────────────────────────────────────────")

total_excluded = (
    len(s1_s2_excluded_ids) +
    len(behavioral_stats["inactive_ids"]) +
    len(behavioral_stats["low_response_ids"]) +
    len(domain_stats["irrelevant_domain_ids"])
)

print(f"\n  TOTAL UNIQUE EXCLUDED (All Stages):  {total_excluded:>6,}  ({total_excluded/total*100:.1f}%)")
print(f"  FINAL CANDIDATES:     {len(final_candidates):>6,}")


[*] Computed AS_OF_DATE for this pool: 2026-05-27

STAGE EXECUTION TIMES
Dynamic Date                  :    0.127 sec
Stage 1 - Exclusion           :    6.139 sec
Stage 2 - Keyword             :    0.702 sec
Stage 3 - Behavioral          :    0.048 sec
Stage 4 - Domain              :    2.970 sec
--------------------------------------------------------------------------------
TOTAL PIPELINE                :    9.985 sec

TOP 30 FUNCTIONS BY CUMULATIVE TIME
         21885784 function calls in 9.985 seconds

   Ordered by: cumulative time
   List reduced from 78 to 30 due to restriction <30>

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
       18    0.000    0.000    9.985    0.555 c:\Users\prami\.conda\envs\hire\lib\site-packages\IPython\core\interactiveshell.py:3543(run_code)
       18    0.000    0.000    9.985    0.555 {built-in method builtins.exec}
        1    0.859    0.859    6.138    6.138 C:\Users\prami\AppData\Local\Temp\ipykernel_13632\2970138449.

In [13]:
jsonl_output_path = '../data/final_candidates_12yoe.jsonl'
with open(jsonl_output_path, 'w') as f:
    for c in final_candidates:
        f.write(json.dumps(c) + '\n')

## Optimized code

In [57]:
TITLE_LEVELS = {
        "junior": 1, "associate": 1, "intern": 1, "trainee": 1,
        "engineer": 2, "developer": 2, "analyst": 2, "scientist": 2,
        "senior": 3, "sr.": 3, "sr ": 3, "lead": 3,
        "staff": 4, "specialist": 4, "expert": 4,
        "principal": 5, "architect": 5,
        "director": 6, "vp": 6, "head": 6, "manager": 6,
    }

In [58]:
ENABLE_PROFILE = True                            # set True for first profiling run
MIN_DOMAIN_GROUPS = 2

In [59]:
# Single OR-regex over all wrong-title keywords
TITLE_PATTERN = re.compile(
    "|".join(re.escape(k) for k in WRONG_TITLE_KEYWORDS),
    re.IGNORECASE,
)

# Single OR-regex over all consulting firm names
CONSULTING_PATTERN = re.compile(
    "|".join(re.escape(f) for f in PURE_CONSULTING_FIRMS),
    re.IGNORECASE,
)

# One compiled pattern per domain group
COMPILED_DOMAIN_GROUPS: dict = {}
for _group, _keywords in DOMAIN_KEYWORD_GROUPS.items():
    _patterns = []
    for _kw in _keywords:
        if _kw[-1].isalnum():
            _patterns.append(r"\b" + re.escape(_kw) + r"\b")
        else:
            _patterns.append(r"\b" + re.escape(_kw))
    COMPILED_DOMAIN_GROUPS[_group] = re.compile("|" .join(_patterns), re.IGNORECASE)

print(f"TITLE_PATTERN    : {len(WRONG_TITLE_KEYWORDS)} keywords compiled")
print(f"CONSULTING_PATTERN: {len(PURE_CONSULTING_FIRMS)} firms compiled")
print(f"DOMAIN_GROUPS    : {list(COMPILED_DOMAIN_GROUPS.keys())}")

TITLE_PATTERN    : 50 keywords compiled
CONSULTING_PATTERN: 20 firms compiled
DOMAIN_GROUPS    : ['core_ml', 'search_ranking', 'embeddings_models', 'vector_dbs', 'evaluation', 'production_systems']


In [60]:
def is_consulting_firm(company_name_lower: str) -> bool:
    """One compiled-regex search replaces looping over the set."""
    return bool(CONSULTING_PATTERN.search(company_name_lower))


def is_pure_consulting_career(companies_lower: list) -> bool:
    """True only if EVERY role in career history is at a consulting firm."""
    if not companies_lower:
        return False
    return all(is_consulting_firm(c) for c in companies_lower)


def is_job_hopper(career: list, companies_lower: list) -> tuple:
    if not career or len(career) < 2:
        return False, ""
    durations  = [r.get("duration_months") or 0 for r in career]
    avg_tenure = sum(durations) / len(durations)
    switches   = len(set(companies_lower))
    if avg_tenure < 18 and switches >= 3:
        return True, f"avg_tenure={avg_tenure:.1f}mo, switches={switches}"
    return False, ""


def _title_level(title_lower: str) -> int:
    matched = [lvl for kw, lvl in TITLE_LEVELS.items() if kw in title_lower]
    return max(matched) if matched else 0


def is_title_chaser(career: list) -> tuple:
    sorted_career = sorted(career, key=lambda r: r.get("start_date") or "0000-00-00")
    jumps, prev_level = 0, 0
    for role in sorted_career:
        level    = _title_level(role.get("title", "").lower())   # lowercase once per role
        duration = role.get("duration_months") or 0
        if level > prev_level and prev_level != 0 and duration < 18:
            jumps += 1
        prev_level = max(prev_level, level)
    if jumps >= 2:
        titles = [r.get("title", "") for r in sorted_career]
        return True, f"quick_jumps={jumps}, titles={titles}"
    return False, ""


def get_dynamic_as_of_date(all_candidates: list) -> date:
    """Snapshot date = max(last_active_date) across the entire pool."""
    max_date_str = None
    for c in all_candidates:
        la = c.get("redrob_signals", {}).get("last_active_date")
        if la and (max_date_str is None or la > max_date_str):
            max_date_str = la
    return date.fromisoformat(max_date_str[:10]) if max_date_str else date.today()


print("Helper functions defined.")

Helper functions defined.


In [61]:
def apply_initial_filters(candidates: list, as_of_date: date) -> tuple:
    """
    Single-pass filter: Stage 1 (structural) + Stage 2 (title) + Stage 3 (behavioural).

    Optimisations applied:
      - Nested dict fields cached as locals once per candidate
      - company names, country, title lowercased once and reused
      - title + consulting firm checks use compiled regex (one .search() each)
      - is_title_chaser only called after cheaper checks pass
      - Stage 3 only runs if Stages 1+2 both passed
    """
    passed = []
    stats  = {
        "non_india_ids":       [],
        "too_junior_ids":      [],
        "too_senior_ids":      [],
        "null_skills_ids":     [],
        "honeypot_skill_ids":  [],
        "pure_consulting_ids": [],
        "job_hopper_ids":      [],
        "title_chaser_ids":    [],
        "wrong_title_ids":     [],
        "inactive_ids":        [],
        "low_response_ids":    [],
        "country_counter":     Counter(),
        "yoe_distribution":    Counter(),
        "consulting_firm_counter": Counter(),
        "title_counter":       Counter(),
        "total_processed":     0,
    }

    for c in candidates:
        stats["total_processed"] += 1

        # ── Cache nested dicts once (Step 4) ─────────────────────────────
        profile  = c.get("profile")  or {}
        career   = c.get("career_history") or []
        signals  = c.get("redrob_signals") or {}
        skills   = c.get("skills") or []
        cid      = c.get("candidate_id", "")

        # ── Lowercase strings once (Step 5) ──────────────────────────────
        country_raw = profile.get("country", "")
        country_l   = country_raw.lower().strip()
        title_raw   = profile.get("current_title", "")
        yoe         = profile.get("years_of_experience", 0) or 0

        # Company names lowercased once — reused for consulting + hopper checks
        companies_l = [r.get("company", "").lower().strip() for r in career]

        # Tracking
        stats["country_counter"][country_raw]         += 1
        stats["yoe_distribution"][(int(yoe) // 2) * 2] += 1
        stats["title_counter"][title_raw]              += 1
        for comp_l in companies_l:
            if is_consulting_firm(comp_l):
                stats["consulting_firm_counter"][comp_l] += 1

        exclude_reasons = []

        # ── Stage 1A: Cheap scalar checks (fail-fast order) ──────────────
        if country_l not in INDIA_VARIANTS:
            if not signals.get("willing_to_relocate", False):
                stats["non_india_ids"].append((cid, country_raw))
                exclude_reasons.append("non_india")

        if yoe < MIN_YOE:
            stats["too_junior_ids"].append((cid, yoe))
            exclude_reasons.append("too_junior")
        elif yoe > MAX_YOE:
            stats["too_senior_ids"].append((cid, yoe))
            exclude_reasons.append("too_senior")

        if len(skills) <= 5:
            stats["null_skills_ids"].append(cid)
            exclude_reasons.append("null_skills")

        # ── Stage 1B: Skills honeypot check ──────────────────────────────
        if skills:
            expert_zero = [
                s for s in skills
                if s.get("proficiency") == "expert"
                and (s.get("duration_months") or 0) == 0
            ]
            if expert_zero:
                stats["honeypot_skill_ids"].append(
                    (cid, [(s["name"], s.get("duration_months")) for s in expert_zero])
                )
                exclude_reasons.append("honeypot_skill")

        # ── Stage 1C: Career structure checks ────────────────────────────
        if is_pure_consulting_career(companies_l):
            stats["pure_consulting_ids"].append((cid, companies_l))
            exclude_reasons.append("pure_consulting")

        hopper, hopper_reason = is_job_hopper(career, companies_l)
        if hopper:
            stats["job_hopper_ids"].append((cid, hopper_reason))
            exclude_reasons.append("job_hopper")

        # title_chaser is the most expensive helper — call it last
        if not exclude_reasons:
            chaser, chaser_reason = is_title_chaser(career)
            if chaser:
                stats["title_chaser_ids"].append((cid, chaser_reason))
                exclude_reasons.append("title_chaser")

        # ── Stage 2: Title keyword pruning (compiled regex, Step 2) ──────
        if TITLE_PATTERN.search(title_raw):
            stats["wrong_title_ids"].append((cid, title_raw))
            exclude_reasons.append("wrong_title")

        if exclude_reasons:
            continue                   # skip Stage 3 entirely

        # ── Stage 3: Behavioural reachability ────────────────────────────
        la_str = signals.get("last_active_date")
        if not la_str:
            stats["inactive_ids"].append((cid, "None"))
            continue

        days_inactive = (as_of_date - date.fromisoformat(la_str[:10])).days
        if days_inactive > MAX_INACTIVE_DAYS:
            stats["inactive_ids"].append((cid, days_inactive))
            continue

        rr = signals.get("recruiter_response_rate")
        if rr is None or rr < MIN_RESPONSE_RATE:
            stats["low_response_ids"].append((cid, rr))
            continue

        passed.append(c)

    return passed, stats


print("apply_initial_filters defined.")

apply_initial_filters defined.


In [62]:
def apply_domain_relevance(candidates: list, min_required_groups: int = 2) -> tuple:
    """
    Ensures the candidate is in the AI/ML/Search domain.
    Checks both skills list and career history descriptions.
    Requires hits in at least `min_required_groups` distinct domain categories.
    """
    passed = []
    stats  = {"irrelevant_domain_ids": [], "total_processed": 0}

    for c in candidates:
        stats["total_processed"] += 1
        cid = c.get("candidate_id", "")

        # Build searchable text — lowercased once
        skill_text = " ".join(
            s.get("name", "").lower() for s in (c.get("skills") or [])
        )
        desc_text = " ".join(
            role.get("description", "").lower()
            for role in (c.get("career_history") or [])
        )
        searchable = f" {skill_text} {desc_text} "

        matched = sum(
            1 for pattern in COMPILED_DOMAIN_GROUPS.values()
            if pattern.search(searchable)
        )

        if matched < min_required_groups:
            stats["irrelevant_domain_ids"].append(cid)
        else:
            passed.append(c)

    return passed, stats


print("apply_domain_relevance defined.")

apply_domain_relevance defined.


In [63]:
def print_summary(total: int, init_stats: dict, domain_stats: dict, final_count: int):
    s = init_stats
    unique_excluded = (
        set(cid for cid, _ in s["non_india_ids"])
        | set(cid for cid, _ in s["too_junior_ids"])
        | set(cid for cid, _ in s["too_senior_ids"])
        | set(s["null_skills_ids"])
        | set(cid for cid, _ in s["honeypot_skill_ids"])
        | set(cid for cid, _ in s["pure_consulting_ids"])
        | set(cid for cid, _ in s["job_hopper_ids"])
        | set(cid for cid, _ in s["title_chaser_ids"])
        | set(cid for cid, _ in s["wrong_title_ids"])
        | set(cid for cid, _ in s["inactive_ids"])
        | set(cid for cid, _ in s["low_response_ids"])
    )

    print("=" * 60)
    print(f"TOTAL CANDIDATES (Initial):       {total:>8,}")
    print("=" * 60)
    print("\n── STAGE 1 & 2: STRUCTURAL & KEYWORD EXCLUSIONS ──")
    rows = [
        (f"Non-India (no relocation)",       len(s["non_india_ids"])),
        (f"Too junior (< {MIN_YOE} yoe)",    len(s["too_junior_ids"])),
        (f"Too senior (> {MAX_YOE} yoe)",    len(s["too_senior_ids"])),
        ("Null/empty skills",                 len(s["null_skills_ids"])),
        ("Honeypot skill signal",             len(s["honeypot_skill_ids"])),
        ("Pure consulting career",            len(s["pure_consulting_ids"])),
        ("Job hopper",                        len(s["job_hopper_ids"])),
        ("Title chaser",                      len(s["title_chaser_ids"])),
        ("Wrong title keyword",               len(s["wrong_title_ids"])),
    ]
    for label, count in rows:
        print(f"  {label:<40} {count:>6,}")

    print("\n── STAGE 3: BEHAVIOURAL ──")
    print(f"  Inactive > {MAX_INACTIVE_DAYS} days                       {len(s['inactive_ids']):>6,}")
    print(f"  Response Rate < {MIN_RESPONSE_RATE*100:.0f}%                          {len(s['low_response_ids']):>6,}")

    print("\n── STAGE 4: DOMAIN RELEVANCE ──")
    print(f"  Lacking core ML/Search depth               {len(domain_stats['irrelevant_domain_ids']):>6,}")

    pct = len(unique_excluded) / total * 100 if total else 0
    print(f"\n  TOTAL UNIQUE EXCLUDED:    {len(unique_excluded):>6,} ({pct:.1f}%)")
    print(f"  FINAL CANDIDATES:         {final_count:>6,}")
    print("=" * 60)


print("print_summary defined.")

print_summary defined.


In [64]:
# ── Derive snapshot date from pool ────────────────────────────────────────────
import sys


as_of_date = get_dynamic_as_of_date(all_candidates)
print(f"AS_OF_DATE (pool max last_active_date): {as_of_date}")

# ── Stage 1 + 2 + 3: Merged single-pass filter ────────────────────────────────
if ENABLE_PROFILE:
    pr = cProfile.Profile()
    pr.enable()

passed_init, init_stats = apply_initial_filters(all_candidates, as_of_date)

if ENABLE_PROFILE:
    pr.disable()
    st = pstats.Stats(pr, stream=sys.stdout)
    st.sort_stats(SortKey.CUMULATIVE)
    print("\n── cProfile: apply_initial_filters ──")
    st.print_stats(20)

print(f"After Stages 1-3: {len(passed_init):,} candidates remain")


# ── Stage 4: Domain relevance ─────────────────────────────────────────────────
if ENABLE_PROFILE:
    pr2 = cProfile.Profile()
    pr2.enable()

final_candidates, domain_stats = apply_domain_relevance(
    passed_init, min_required_groups=MIN_DOMAIN_GROUPS
)

if ENABLE_PROFILE:
    pr2.disable()
    st2 = pstats.Stats(pr2, stream=sys.stdout)
    st2.sort_stats(SortKey.CUMULATIVE)
    print("\n── cProfile: apply_domain_relevance ──")
    st2.print_stats(15)

print(f"After Stage 4:    {len(final_candidates):,} candidates remain")

AS_OF_DATE (pool max last_active_date): 2026-05-27

── cProfile: apply_initial_filters ──
         7256167 function calls in 4.565 seconds

   Ordered by: cumulative time
   List reduced from 45 to 20 due to restriction <20>

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
        2    0.000    0.000    4.585    2.293 c:\Users\prami\.conda\envs\hire\lib\site-packages\IPython\core\interactiveshell.py:3543(run_code)
        2    0.000    0.000    4.585    2.293 {built-in method builtins.exec}
        1    0.869    0.869    4.565    4.565 C:\Users\prami\AppData\Local\Temp\ipykernel_24528\4282575929.py:1(apply_initial_filters)
   531395    1.181    0.000    1.181    0.000 {method 'search' of 're.Pattern' objects}
   431515    0.130    0.000    0.929    0.000 C:\Users\prami\AppData\Local\Temp\ipykernel_24528\1511824533.py:1(is_consulting_firm)
    49236    0.200    0.000    0.793    0.000 C:\Users\prami\AppData\Local\Temp\ipykernel_24528\1511824533.py:29(is_title_cha

In [ ]:
USE_PARALLEL = True        # ← flip to True to enable
NUM_WORKERS  = None         # None → auto-detect (min(cpu_count, 8))

def _merge_init_stats(results: list) -> dict:
    merged = {
        "non_india_ids": [], "too_junior_ids": [], "too_senior_ids": [],
        "null_skills_ids": [], "honeypot_skill_ids": [], "pure_consulting_ids": [],
        "job_hopper_ids": [], "title_chaser_ids": [], "wrong_title_ids": [],
        "inactive_ids": [], "low_response_ids": [],
        "country_counter": Counter(), "yoe_distribution": Counter(),
        "consulting_firm_counter": Counter(), "title_counter": Counter(),
        "total_processed": 0,
    }
    list_keys = [
        "non_india_ids", "too_junior_ids", "too_senior_ids", "null_skills_ids",
        "honeypot_skill_ids", "pure_consulting_ids", "job_hopper_ids",
        "title_chaser_ids", "wrong_title_ids", "inactive_ids", "low_response_ids",
    ]
    counter_keys = [
        "country_counter", "yoe_distribution",
        "consulting_firm_counter", "title_counter",
    ]
    for _, stats in results:
        merged["total_processed"] += stats["total_processed"]
        for k in list_keys:
            merged[k].extend(stats[k])
        for k in counter_keys:
            merged[k].update(stats[k])
    return merged


def run_parallel(candidates: list, as_of_date: date,
                 min_domain_groups: int = 2, num_workers=None):
    """Run apply_initial_filters + apply_domain_relevance in parallel via joblib."""
    try:
        from joblib import Parallel, delayed
        import os
    except ImportError:
        print("joblib not installed — falling back to serial. `pip install joblib` to enable.")
        passed_init, init_stats = apply_initial_filters(candidates, as_of_date)
        final_, dom_stats = apply_domain_relevance(passed_init, min_domain_groups)
        return passed_init, init_stats, final_, dom_stats

    n_workers = num_workers or min(os.cpu_count() or 4, 8)
    if len(candidates) < 5000 or n_workers <= 1:
        passed_init, init_stats = apply_initial_filters(candidates, as_of_date)
        final_, dom_stats = apply_domain_relevance(passed_init, min_domain_groups)
        return passed_init, init_stats, final_, dom_stats

    chunk = (len(candidates) + n_workers - 1) // n_workers
    chunks = [candidates[i:i+chunk] for i in range(0, len(candidates), chunk)]

    print(f"  → Running initial filters on {n_workers} workers, {chunk:,} candidates each")
    init_results = Parallel(n_jobs=n_workers, backend="loky")(
        delayed(apply_initial_filters)(ch, as_of_date) for ch in chunks
    )
    passed_init = [c for passed, _ in init_results for c in passed]
    init_stats  = _merge_init_stats(init_results)

    print(f"  → Running domain relevance on {n_workers} workers")
    dom_chunks = [passed_init[i:i+chunk] for i in range(0, len(passed_init), chunk)] or [[]]
    dom_results = Parallel(n_jobs=n_workers, backend="loky")(
        delayed(apply_domain_relevance)(ch, min_domain_groups) for ch in dom_chunks
    )
    final_ = [c for passed, _ in dom_results for c in passed]
    dom_stats = {"irrelevant_domain_ids": [], "total_processed": 0}
    for _, s in dom_results:
        dom_stats["irrelevant_domain_ids"].extend(s["irrelevant_domain_ids"])
        dom_stats["total_processed"] += s["total_processed"]

    return passed_init, init_stats, final_, dom_stats


if USE_PARALLEL:
    import time
    t0 = time.perf_counter()
    passed_init, init_stats, final_candidates, domain_stats = run_parallel(
        all_candidates, as_of_date,
        min_domain_groups=MIN_DOMAIN_GROUPS,
        num_workers=NUM_WORKERS,
    )
    print(f"Parallel pipeline finished in {time.perf_counter()-t0:.2f}s")
    print(f"After Stages 1-3: {len(passed_init):,} candidates remain")
    print(f"After Stage 4:    {len(final_candidates):,} candidates remain")
else:
    print("USE_PARALLEL = False — using serial pipeline from cells 21 and 22")


  → Running initial filters on 8 workers, 12,485 candidates each
  → Running domain relevance on 8 workers
Parallel pipeline finished in 26.84s
After Stages 1-3: 19,219 candidates remain
After Stage 4:    4,896 candidates remain


In [66]:
# ── Summary ───────────────────────────────────────────────────────────────────
print_summary(len(all_candidates), init_stats, domain_stats, len(final_candidates))

TOTAL CANDIDATES (Initial):         99,880

── STAGE 1 & 2: STRUCTURAL & KEYWORD EXCLUSIONS ──
  Non-India (no relocation)                17,735
  Too junior (< 3 yoe)                     16,105
  Too senior (> 12 yoe)                    14,369
  Null/empty skills                         5,545
  Honeypot skill signal                        21
  Pure consulting career                    9,737
  Job hopper                                1,508
  Title chaser                                  0
  Wrong title keyword                      63,049

── STAGE 3: BEHAVIOURAL ──
  Inactive > 180 days                          658
  Response Rate < 10%                             154

── STAGE 4: DOMAIN RELEVANCE ──
  Lacking core ML/Search depth               14,323

  TOTAL UNIQUE EXCLUDED:    80,661 (80.8%)
  FINAL CANDIDATES:          4,896
